# Create a hold-out set for final testing

Author: Pete King

This notebook creates a separate testing dataset to hold in reserve for a final evaluation of our selected model's ability to generalize to unseen data.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import re

import pandas as pd

import data_prep as dp

In [2]:
# Select the data file and start date for the testing set
# ----------------------------------------------------------------------------
# We want to rigorously test our final model against a period of dynamic
# real-world volatility.  For example, beginning in October 2024 would capture
# an initial period of relatively low volatility, followed by the highly 
# volatility period after Trump announced reciprocal tarrifs in April 2025.
# ----------------------------------------------------------------------------
DATA_FILENAME = 'raw_data_prediction_dataset.csv'
TESTING_START_DATE = '2024-10-01'
# Select buffer size between validation/test and training sets
# We calculate target variables using 63 days of "future" historical data
# A buffer of one quarter (63 business days) guards against data leakage
BUFFER_SIZE = 63

In [3]:
# Import feature dataset with labels
df = pd.read_csv(DATA_FILENAME)
df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28523,2026-03-05,31490.07,24111.83,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28524,2026-03-06,31490.07,24111.83,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28525,2026-03-09,31490.07,24111.83,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28526,2026-03-10,31490.07,24111.83,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Let's look at the earliest starting dates for each of our column labels to see how far back our labels go for each ETF.

In [4]:
label_tag = '_volatility_target'
# Use a regular expression to parse the ticker symbol from the column name
p = re.compile(label_tag)
start_dates = {}
for column_name in df.columns:
    if label_tag in column_name:
        ticker = p.split(column_name)[0]
        start_dates[ticker] = (
            df[['date', column_name]].dropna().date.iloc[0]
        )
sorted([(k, v) for k, v in start_dates.items()], key=lambda pair: pair[1])

[('XLF', '1998-12-22'),
 ('XLK', '1998-12-22'),
 ('XLU', '1998-12-22'),
 ('XLV', '1998-12-22'),
 ('XLE', '1998-12-22'),
 ('XLI', '1998-12-22'),
 ('XLB', '1998-12-22'),
 ('XLP', '1998-12-22'),
 ('XLY', '1998-12-22'),
 ('IEF', '2002-07-30'),
 ('TLT', '2002-07-30'),
 ('LQD', '2002-07-30'),
 ('TIP', '2003-12-05'),
 ('GLD', '2004-11-18'),
 ('HYG', '2007-04-11'),
 ('BIL', '2007-05-30'),
 ('XLRE', '2015-10-08')]

In [5]:
print(f'The XLRE (Real Estate) series only goes back to {start_dates['XLRE']}')

The XLRE (Real Estate) series only goes back to 2015-10-08


For the Vector AutoRegressive Integrated Moving Average (VARIMA) and the multivariate Long Short-Term Memory (LSTM) models, we intend to use fluctuations in the price of all labeled ETF price time series as features.  However, we've observed that a few of these time series have significantly later start dates than the others.  For example, if we drop the 'XLRE' ETF (Real Estate) from the dataset, we'll free up 8 years worth of training data for the rest of the ETFs.

Going forward, we plan to test three variations of the data:

 - Including all raw data from roughly 2015 (start of XLRE data)
 - Dropping XLRE (Real Estate) to make data as early as 2007 available
 - Dropping HYG and BIL to make data as early as 2004 available

In [6]:
train_val_df, test_df = dp.time_series_split(
    df, TESTING_START_DATE, BUFFER_SIZE
)
train_val_df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28006,2024-07-15,29511.664,23478.57,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,0.090963,0.224980,0.125842,0.002766,0.056791,0.124879,0.053551,0.034949,0.036434,0.135357
28007,2024-07-16,29511.664,23478.57,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,0.089399,0.222424,0.127014,0.002766,0.056333,0.122062,0.052568,0.034383,0.036206,0.130409
28008,2024-07-17,29511.664,23478.57,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,0.086208,0.219621,0.126133,0.002766,0.056739,0.122378,0.052992,0.034429,0.036437,0.131185
28009,2024-07-18,29511.664,23478.57,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,0.085397,0.218482,0.124614,0.002784,0.056316,0.121168,0.052032,0.034033,0.036048,0.130300


In [7]:
test_df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
28074,2024-10-01,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,0.087169,0.162650,0.139499,0.002395,0.057190,0.135787,0.067372,0.029942,0.037312,0.150887
28075,2024-10-02,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,0.085636,0.161105,0.139556,0.002421,0.057603,0.136229,0.068072,0.030536,0.037614,0.152870
28076,2024-10-03,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,0.083165,0.158537,0.138286,0.002435,0.056876,0.135196,0.066973,0.030132,0.036780,0.152865
28077,2024-10-04,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,0.083609,0.158616,0.138042,0.002432,0.053749,0.133427,0.066250,0.030237,0.034352,0.152894
28078,2024-10-05,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,0.083682,0.158503,0.138287,0.002415,0.053749,0.133572,0.066252,0.030579,0.034356,0.153087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28523,2026-03-05,31490.070,24111.830,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28524,2026-03-06,31490.070,24111.830,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28525,2026-03-09,31490.070,24111.830,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28526,2026-03-10,31490.070,24111.830,122.30525,1227.495,326.588,332.793,128.605,127.918,152.174,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The gap between the training and testing sets guards against data leakage.

In [8]:
# Save segmented datasets for further analysis; drop the index
train_val_df.to_csv('train_val_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)